# 토큰화 리스트를 이용하여 BERTopic 분석 진행


## 파일 특징

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("df_tokens.parquet")

# 컬럼 타입 및 PBSH 샘플
print("[컬럼 타입]")
print(df.dtypes)
print("\n[PBSH 샘플 10개]")
print(df["PBSH"].head(10))

[컬럼 타입]
NODE_ID                object
PBSH                   object
NODE_CLSS_02           object
NODE_TTLE              object
NODE_TTLE_EN           object
ABST_KR                object
ABST_EN                object
KYWD                   object
ko_text                object
en_text                object
ko_tokens              object
en_tokens              object
merged_tokens_dedup    object
dtype: object

[PBSH 샘플 10개]
0    202106
1    202106
2    202106
3    202106
4    202106
5    202106
6    202106
7    202106
8    202106
9    202106
Name: PBSH, dtype: object


In [3]:
# 중분류 분포
print('\n')
print(df['NODE_CLSS_02'].value_counts())



NODE_CLSS_02
전기전자공학      17809
기계공학        12278
건축공학        11192
컴퓨터학         6703
공학 일반        5517
기타 공학        1948
화학공학         1400
재료·에너지공학     1365
산업공학         1361
조선해양공학       1134
Name: count, dtype: int64


In [4]:
# 전체 행 수와 기간별 분포
print("\n전체 데이터 크기:", len(df))
print("\n연도별 분포:")
df['year'] = df['PBSH'].str[:4]
print(df['year'].value_counts().sort_index())

print("\n[merged_tokens_dedup 샘플 5개 + 타입]")
for i in range(5):
    val = df["merged_tokens_dedup"].iloc[i]
    print(type(val), val[:10] if isinstance(val,(list,np.ndarray)) else val)



전체 데이터 크기: 60707

연도별 분포:
year
2021    12435
2022    12333
2023    12203
2024    12658
2025    11078
Name: count, dtype: int64

[merged_tokens_dedup 샘플 5개 + 타입]
<class 'numpy.ndarray'> ['카테시안' '공간' '시간' '지연제어' '튜토리얼' 'robot control' 'robot manipulators'
 'time-delayed control' 'cartesian space control' 'tutorial']
<class 'numpy.ndarray'> ['파이프' '구조물' '검사' '로봇' '위치' '추적' 'pipe-climbing' 'pipe climbing robot'
 'location tracking' 'position']
<class 'numpy.ndarray'> ['depth' 'amplitude' 'image' '차량' '주변' '장애물' '인식' '거리' '추정'
 'tof (time of flight) camera']
<class 'numpy.ndarray'> ['단시간' '푸리에' '변환' '전역' '통과' '필터' '노치' '다중' '주파수' '제거']
<class 'numpy.ndarray'> ['mems' 'fog' '관성항법' '장치' '단계' '교정' '개발' 'mems/fog'
 'inertial navigation system' 'two stage calibration technique']


# 딥러닝 및 머신러닝 BERTopic 분석 시작
- 토큰화가 완료된 df_tokens.parquet 파일 사용
- PBSH에서 year 추출
- KYWD/제목/초록으로 ai_type 자동 분류

## 패키지 설치

In [5]:
# 패키지 설치
!pip install bertopic sentence-transformers umap-learn hdbscan pyarrow -q

# 한글 폰트 설치
!apt-get -qq -y install fonts-nanum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.1 MB/s eta 0:00:00
Selecting previously unselected package fonts-nanum.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


## 임포트

In [6]:
import sys
import subprocess

def install_packages():
    packages = [
        'bertopic',
        'sentence-transformers',
        'umap-learn',
        'hdbscan',
        'pyarrow'
    ]

    print("자동으로 패키지 설치")
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
        except ImportError:
            print(f"{package} 설치 중...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
    print("설치\n")

# Colab 환경인 경우 패키지 설치
try:
    import google.colab
    IN_COLAB = True
    install_packages()
except:
    IN_COLAB = False


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

# BERTopic 관련
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# 한글 폰트 설정
import platform
system = platform.system()

if IN_COLAB:
    # Colab용 한글 폰트 설정
    subprocess.run(['apt-get', '-qq', '-y', 'install', 'fonts-nanum'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    import matplotlib.font_manager as fm
    fontpath = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
    font = fm.FontProperties(fname=fontpath, size=10)
    plt.rc('font', family='NanumBarunGothic')
elif system == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif system == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False

if IN_COLAB:
    print("한글 폰트 설정\n")

자동으로 패키지 설치


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


umap-learn 설치 중...
설치

한글 폰트 설정



## 1. AI 분류 패턴 정의

In [7]:
TRADITIONAL_ML_PATTERNS = [
    r'\bml\b', r'machine\s+learning', r'supervised\s+learning', r'unsupervised\s+learning',
    r'\bsvm\b', r'support\s+vector', r'random\s+forest', r'decision\s+tree',
    r'k-?nearest', r'\bknn\b', r'naive\s+bayes', r'logistic\s+regression',
    r'gradient\s+boosting', r'\bxgboost\b', r'\blightgbm\b',
    r'머신\s?러닝', r'기계\s?학습', r'지도\s?학습', r'비지도\s?학습',
]

DEEP_LEARNING_PATTERNS = [
    r'deep\s+learning', r'neural\s+network', r'\bcnn\b', r'\brnn\b', r'\blstm\b',
    r'\bgru\b', r'transformer', r'attention', r'\bbert\b', r'\bgpt\b',
    r'computer\s+vision', r'object\s+detection', r'\bgan\b', r'autoencoder',
    r'resnet', r'yolo', r'vgg', r'alexnet', r'inception',
    r'딥\s?러닝', r'심층\s?학습', r'신경망', r'합성곱', r'순환\s?신경망',
    r'트랜스포머', r'어텐션', r'객체\s?탐지', r'이미지\s?분류',
]

# 논문을 '전통ML' 또는 '딥러닝'으로 분류
def classify_paper(row):
    # KYWD, 제목, 초록을 모두 합쳐서 검색
    kywd = str(row.get('KYWD', '')) if pd.notna(row.get('KYWD')) else ''
    title = str(row.get('NODE_TTLE_EN', '')) if pd.notna(row.get('NODE_TTLE_EN')) else ''
    abstract = str(row.get('ABST_EN', '')) if pd.notna(row.get('ABST_EN')) else ''

    text = (kywd + ' ' + title + ' ' + abstract).lower()

    # 딥러닝 먼저 체크 (더 세분화된 분류이므로)
    for pattern in DEEP_LEARNING_PATTERNS:
        if re.search(pattern, text, re.IGNORECASE):
            return '딥러닝'

    # 전통ML 체크
    for pattern in TRADITIONAL_ML_PATTERNS:
        if re.search(pattern, text, re.IGNORECASE):
            return '전통ML'

    return None


## 2. 불용어 정의

In [8]:
KOREAN_STOPWORDS = [
    # 일반 불용어
    '것', '수', '등', '및', '이', '그', '저', '개', '명', '번', '위',
    '때', '년', '월', '일', '시', '분', '초', '곳', '데', '점',

    # 대명사
    '나', '너', '우리', '저희', '당신', '여러분', '누구', '무엇', '어디',

    # 의존명사
    '것', '거', '수', '때', '데', '바', '줄', '적', '번', '차', '대로',

    # 일반적인 연구 용어
    '연구', '분석', '결과', '방법', '제안', '제시', '개발', '설계',
    '적용', '활용', '이용', '사용', '구현', '시스템', '기술', '방식',
    '과정', '단계', '요소', '특징', '문제', '해결', '성능', '효과',
    '효율', '정확', '정확도', '비교', '평가', '실험', '검증', '측정',
    '향상', '개선', '최적', '최적화', '가능', '필요', '중요', '기반',
    '통해', '위해', '대해', '대한', '관련', '따른', '위한', '통한',
    '각', '각각', '모든', '여러', '다양', '주요', '전체', '일반',
    '기존', '새로운', '다른', '같은', '이러한', '그러한',

    # AI/ML 분류 키워드 (결과에 나오면 안됨!)
    '딥러닝', '심층', '학습', '신경망', '합성곱', '순환',
    '머신러닝', '기계학습', '지도학습', '비지도학습', '강화학습',
    '인공지능', 'ai', '트랜스포머', '어텐션',

    # 데이터 관련 일반 용어
    '데이터', '훈련', '테스트', '검증', '샘플', '입력', '출력',
]

ENGLISH_STOPWORDS = [
    # 기본 불용어
    'the', 'of', 'and', 'to', 'in', 'is', 'it', 'for', 'as', 'was', 'with',
    'be', 'by', 'on', 'at', 'from', 'or', 'an', 'are', 'this', 'that', 'which',
    'their', 'we', 'our', 'these', 'those', 'than', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'between', 'under', 'again', 'further',
    'also', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such',
    'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'then', 'too', 'very',
    'can', 'will', 'just', 'should', 'now',

    # AI/ML 분류 키워드 (결과에 나오면 안됨!)
    'deep', 'learning', 'machine', 'neural', 'network', 'networks',
    'cnn', 'rnn', 'lstm', 'gru', 'transformer', 'transformers',
    'ml', 'dl', 'ai', 'artificial', 'intelligence',

    # 너무 일반적인 ML 용어
    'data', 'model', 'models', 'modeling', 'training', 'train', 'trained',
    'test', 'testing', 'validation', 'dataset', 'datasets',
    'input', 'output', 'feature', 'features', 'parameter', 'parameters',
    'layer', 'layers', 'epoch', 'epochs', 'batch', 'batches',

    # 시간/크기 관련
    'long', 'short', 'large', 'small', 'big', 'term', 'terms',
    'pre', 'post', 'multi', 'single', 'double',

    # 너무 일반적인 동사
    'based', 'using', 'used', 'use', 'uses',
    'make', 'makes', 'making', 'made',
    'show', 'shows', 'shown', 'showing',
    'propose', 'proposed', 'proposing',
    'present', 'presented', 'presenting',
    'provide', 'provides', 'provided', 'providing',
    'achieve', 'achieved', 'achieving',
    'improve', 'improved', 'improving', 'improvement', 'improvements',
    'demonstrate', 'demonstrated', 'demonstrating',
    'obtain', 'obtained', 'obtaining',
    'evaluate', 'evaluated', 'evaluating', 'evaluation',

    # 연구 관련 일반 용어
    'method', 'methods', 'methodology',
    'study', 'studies', 'research',
    'analysis', 'analyze', 'analyzed',
    'approach', 'approaches',
    'technique', 'techniques',
    'algorithm', 'algorithms',
    'system', 'systems',
    'application', 'applications', 'applied',
    'experiment', 'experiments', 'experimental',
    'paper', 'work', 'works',

    # 결과/성능 관련
    'performance', 'result', 'results',
    'accuracy', 'accurate', 'precisely', 'precision',
    'efficient', 'efficiency', 'effectively', 'effective',
    'better', 'best', 'good', 'well',
    'high', 'higher', 'highest',
    'low', 'lower', 'lowest',
    'fast', 'faster', 'fastest',
    'robust', 'robustness',

    # 비교/평가
    'compare', 'compared', 'comparison', 'comparisons',
    'different', 'difference', 'differences',
    'various', 'variety',
    'several', 'multiple',

    # 논리/연결어
    'however', 'therefore', 'thus', 'hence',
    'consider', 'considered', 'considering',
    'given', 'since', 'while', 'although',

    # 수량/순서
    'number', 'numbers',
    'one', 'two', 'three', 'four', 'five',
    'first', 'second', 'third',
    'may', 'might', 'could', 'would', 'should',

    # 목적/방식
    'way', 'ways', 'manner',
    'need', 'needs', 'needed', 'requiring', 'required',
    'important', 'significance', 'significant',
    'main', 'major', 'key', 'primary',
    'general', 'specific', 'particular',
]

ALL_STOPWORDS = KOREAN_STOPWORDS + ENGLISH_STOPWORDS

## 3. 데이터 로드 및 분류

In [9]:
def load_and_classify_papers(parquet_file):

    df = pd.read_parquet(parquet_file)
    print(f"{len(df):,}개 논문 로드 완료")

    # PBSH에서 year 추출 및 AI 유형 분류
    print("연도 정보 추출 및 AI 유형 분류")
    df['year'] = df['PBSH'].str[:4]
    df['ai_type'] = df.apply(classify_paper, axis=1)

    df_classified = df[df['ai_type'].notna()].copy()

    print(f"전체: {len(df):,}편")
    print(f"분류됨: {len(df_classified):,}편 ({len(df_classified)/len(df)*100:.1f}%)")
    print(f"딥러닝: {(df_classified['ai_type'] == '딥러닝').sum():,}편")
    print(f"전통ML: {(df_classified['ai_type'] == '전통ML').sum():,}편")

    papers_by_type_period = {
        '전통ML': {'2021-2022': [], '2023-2025': []},
        '딥러닝': {'2021-2022': [], '2023-2025': []}
    }

    for ai_type in ['전통ML', '딥러닝']:

        for period in ['2021-2022', '2023-2025']:
            # 기간별 필터링
            if period == '2021-2022':
                mask = (df_classified['ai_type'] == ai_type) & (df_classified['year'].isin(['2021', '2022']))
            else:
                mask = (df_classified['ai_type'] == ai_type) & (df_classified['year'].isin(['2023', '2024', '2025']))

            filtered = df_classified[mask]

            # 토큰을 문자열로 변환
            for _, row in filtered.iterrows():
                tokens = row['merged_tokens_dedup']

                # numpy array를 list로 변환 후 문자열로
                if isinstance(tokens, np.ndarray):
                    tokens = tokens.tolist()

                # 토큰 필터링
                filtered_tokens = []
                for token in tokens:
                    token_str = str(token).strip()
                    # 너무 짧은 토큰 제거 (1글자)
                    if len(token_str) < 2:
                        continue
                    # 순수 숫자만 있는 토큰 제거
                    if token_str.isdigit():
                        continue
                    # 특수문자만 있는 토큰 제거
                    if not any(c.isalnum() for c in token_str):
                        continue
                    filtered_tokens.append(token_str)

                # BERTopic 입력용 텍스트
                text = ' '.join(filtered_tokens)

                # 최소 길이 체크
                if len(filtered_tokens) >= 10:
                    papers_by_type_period[ai_type][period].append({
                        'text': text,
                        'tokens': filtered_tokens,
                        'year': row['year'],
                        'title': row.get('NODE_TTLE_EN', ''),
                        'keywords': row.get('KYWD', '')
                    })

    # 통계 출력
    print("-" * 70)
    for ai_type in ['전통ML', '딥러닝']:
        before = len(papers_by_type_period[ai_type]['2021-2022'])
        after = len(papers_by_type_period[ai_type]['2023-2025'])
        print(f"\n{ai_type}:")
        print(f"  2021-2022: {before:,}편")
        print(f"  2023-2025: {after:,}편")
        print(f"  총: {before + after:,}편")

    return papers_by_type_period

## 4. BERTopic 모델 생성
- AI 유형에 맞춘 BERTopic 모델 생성
- 전통ML: 데이터가 적으므로 클러스터 크기 감소
- 딥러닝: 표준 파라미터

In [10]:
def create_bertopic_model(ai_type='딥러닝', n_topics=None):

    # 임베딩 모델: Multilingual
    embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

    # UMAP
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric='cosine',
        random_state=42
    )

    # HDBSCAN (AI 유형별 파라미터 조정)
    if ai_type == '전통ML':
        min_cluster = 10  # 데이터가 적음
        min_samples = 3
        print(f"{ai_type}용 파라미터: min_cluster={min_cluster}, min_samples={min_samples}")
    else:
        min_cluster = 15  # 데이터가 상대적으로 많음
        min_samples = 5
        print(f"{ai_type}용 파라미터: min_cluster={min_cluster}, min_samples={min_samples}")

    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster,
        min_samples=min_samples,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )

    # CountVectorizer (파라미터 강화)
    vectorizer_model = CountVectorizer(
        ngram_range=(1, 2),
        stop_words=ALL_STOPWORDS,
        min_df=2,       # 최소 2개 문서에 나타나야 함
        max_df=0.90,    # 90% 이상 문서에 나타나는 단어 제외
        max_features=None,
        lowercase=False
    )

    # BERTopic 모델
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        nr_topics=n_topics,
        top_n_words=15,
        verbose=False,
        calculate_probabilities=True
    )

    return topic_model

## 5. BERTopic 분석 실행

In [11]:
def analyze_with_bertopic(papers_by_type_period, n_topics='auto'):

    results = {}

    for ai_type in ['전통ML', '딥러닝']:
        results[ai_type] = {}

        for period in ['2021-2022', '2023-2025']:
            papers = papers_by_type_period[ai_type][period]

            # 최소 논문 수 체크
            min_papers = 30 if ai_type == '딥러닝' else 20
            if len(papers) < min_papers:
                print(f"{ai_type} ({period}): 논문 수 부족 ({len(papers)}편), skip")
                continue

            print(f"{ai_type} ({period}) BERTopic 분석")
            print(f"논문 수: {len(papers):,}편")

            documents = [p['text'] for p in papers]

            # n_topics 설정
            if n_topics == 'auto':
                num_topics = None
            else:
                num_topics = min(n_topics, len(papers) // 20)

            # BERTopic 모델 생성
            topic_model = create_bertopic_model(
                ai_type=ai_type,
                n_topics=num_topics
            )

            # 토픽 추출
            topics, probs = topic_model.fit_transform(documents)

            results[ai_type][period] = {
                'model': topic_model,
                'topics': topics,
                'probs': probs,
                'documents': documents,
                'papers': papers
            }

            # 토픽 정보
            topic_info = topic_model.get_topic_info()
            n_topics_found = len(topic_info[topic_info['Topic'] != -1])
            outliers = (topic_info[topic_info['Topic'] == -1]['Count'].values[0]
                       if -1 in topic_info['Topic'].values else 0)

            print(f"토픽 수: {n_topics_found}개")
            print(f"아웃라이어: {outliers}개 ({outliers/len(papers)*100:.1f}%)")

            print(f"\n상위 5개 토픽:")
            for idx, row in topic_info[topic_info['Topic'] != -1].head(5).iterrows():
                topic_id = row['Topic']
                topic_words = topic_model.get_topic(topic_id)[:5]
                words = [word for word, score in topic_words]
                count = row['Count']
                print(f"   Topic {topic_id} ({count}편): {', '.join(words)}")
            print(f"\n{'━'*70}\n")
    return results

## 6. 시각화

In [12]:
def visualize_topics(results, output_dir='bertopic_results'):

    import os
    os.makedirs(output_dir, exist_ok=True)

    for ai_type in ['전통ML', '딥러닝']:
        for period in ['2021-2022', '2023-2025']:
            if period not in results[ai_type]:
                continue

            model = results[ai_type][period]['model']

            print(f"\n{ai_type} ({period}):")

            # 막대 그래프
            try:
                fig = model.visualize_barchart(top_n_topics=8, n_words=10)
                filename = f"{output_dir}/{ai_type}_{period}_barchart.html".replace(' ', '_')
                fig.write_html(filename)
                print(f"막대 그래프: {filename}")
            except Exception as e:
                print(f"막대 실패: {str(e)[:50]}")

            # 토픽 거리 맵
            try:
                fig = model.visualize_topics()
                filename = f"{output_dir}/{ai_type}_{period}_topics.html".replace(' ', '_')
                fig.write_html(filename)
                print(f"토픽 맵: {filename}")
            except Exception as e:
                print(f"토픽 실패: {str(e)[:50]}")

            # 계층 구조
            try:
                fig = model.visualize_hierarchy()
                filename = f"{output_dir}/{ai_type}_{period}_hierarchy.html".replace(' ', '_')
                fig.write_html(filename)
                print(f"계층 구조: {filename}")
            except Exception as e:
                print(f"계층 실패: {str(e)[:50]}")

## 7. 토픽 비교

In [13]:
def compare_topics(results):

    print("\n")
    print("기간별 토픽 비교")

    for ai_type in ['전통ML', '딥러닝']:
        if '2021-2022' not in results[ai_type] or '2023-2025' not in results[ai_type]:
            print(f"\n{ai_type}: 비교할 데이터 부족")
            continue

        print(f"\n{'━'*70}")
        print(f"{ai_type} 분야")
        print('━'*70)

        model_before = results[ai_type]['2021-2022']['model']
        model_after = results[ai_type]['2023-2025']['model']

        # 2021-2022
        print(f"\n2021-2022 주요 토픽:")
        topics_before = model_before.get_topic_info()
        for idx, row in topics_before[topics_before['Topic'] != -1].head(5).iterrows():
            topic_id = row['Topic']
            topic_words = model_before.get_topic(topic_id)[:5]
            words = [word for word, score in topic_words]
            count = row['Count']
            print(f"   Topic {topic_id} ({count}편): {', '.join(words)}")

        # 2023-2025
        print(f"\n2023-2025 주요 토픽:")
        topics_after = model_after.get_topic_info()
        for idx, row in topics_after[topics_after['Topic'] != -1].head(5).iterrows():
            topic_id = row['Topic']
            topic_words = model_after.get_topic(topic_id)[:5]
            words = [word for word, score in topic_words]
            count = row['Count']
            print(f"   Topic {topic_id} ({count}편): {', '.join(words)}")

## 8. 엑셀 저장

In [14]:
def save_results(results, output_file='bertopic_analysis.xlsx'):

    print(f"\n결과 저장: {output_file}")

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:

        # 요약 시트
        summary_data = []
        for ai_type in ['전통ML', '딥러닝']:
            for period in ['2021-2022', '2023-2025']:
                if period in results[ai_type]:
                    model = results[ai_type][period]['model']
                    topic_info = model.get_topic_info()
                    n_topics = len(topic_info[topic_info['Topic'] != -1])
                    n_docs = len(results[ai_type][period]['documents'])
                    outliers = (topic_info[topic_info['Topic'] == -1]['Count'].values[0]
                               if -1 in topic_info['Topic'].values else 0)

                    summary_data.append({
                        'AI유형': ai_type,
                        '기간': period,
                        '논문수': n_docs,
                        '토픽수': n_topics,
                        '아웃라이어': outliers,
                        '아웃라이어비율(%)': f"{outliers/n_docs*100:.1f}"
                    })

        df_summary = pd.DataFrame(summary_data)
        df_summary.to_excel(writer, sheet_name='요약', index=False)

        # 각 토픽 상세 정보
        for ai_type in ['전통ML', '딥러닝']:
            for period in ['2021-2022', '2023-2025']:
                if period not in results[ai_type]:
                    continue

                model = results[ai_type][period]['model']
                topic_info = model.get_topic_info()
                topic_info = topic_info[topic_info['Topic'] != -1].copy()

                # 상위 단어 추출
                topic_words_list = []
                for topic_id in topic_info['Topic']:
                    topic_words = model.get_topic(topic_id)[:10]
                    words = ', '.join([f"{word}({score:.3f})" for word, score in topic_words])
                    topic_words_list.append(words)

                topic_info['Top_Words'] = topic_words_list

                # 시트 이름 (최대 31자)
                sheet_name = f"{ai_type}_{period}".replace('/', '-')[:31]
                topic_info.to_excel(writer, sheet_name=sheet_name, index=False)

# 메인

In [15]:
PARQUET_FILE = 'df_tokens.parquet'
OUTPUT_DIR = 'bertopic_results'
N_TOPICS = 'auto'

## 데이터 로드 및 분류

In [16]:
papers_by_type_period = load_and_classify_papers(PARQUET_FILE)

print("데이터 로드 및 분류 완료")
print({k: len(v) for k, v in papers_by_type_period.items()})

60,707개 논문 로드 완료
연도 정보 추출 및 AI 유형 분류
전체: 60,707편
분류됨: 9,492편 (15.6%)
딥러닝: 7,830편
전통ML: 1,662편
----------------------------------------------------------------------

전통ML:
  2021-2022: 661편
  2023-2025: 995편
  총: 1,656편

딥러닝:
  2021-2022: 2,910편
  2023-2025: 4,904편
  총: 7,814편
데이터 로드 및 분류 완료
{'전통ML': 2, '딥러닝': 2}


## BERTopic 분석

In [17]:
results = analyze_with_bertopic(papers_by_type_period, n_topics=N_TOPICS)

전통ML (2021-2022) BERTopic 분석
논문 수: 661편


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

전통ML용 파라미터: min_cluster=10, min_samples=3
토픽 수: 17개
아웃라이어: 162개 (24.5%)

상위 5개 토픽:
   Topic 0 (77편): cell, water, concentration, activity, respectively
   Topic 1 (72편): technology, 확인, 제공, 분야, level
   Topic 2 (53편): health, survey, logistic, logistic regression, 회귀분석
   Topic 3 (43편): power, energy, generation, forecasting, distribution
   Topic 4 (33편): detection, 탐지, attack, 공격, maliciou

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

전통ML (2023-2025) BERTopic 분석
논문 수: 995편
전통ML용 파라미터: min_cluster=10, min_samples=3
토픽 수: 26개
아웃라이어: 301개 (30.3%)

상위 5개 토픽:
   Topic 0 (96편): health, survey, logistic regression, logistic, 회귀분석
   Topic 1 (50편): detection, security, malware, 탐지, attack
   Topic 2 (46편): robot, unity, reinforcement, virtual, environment
   Topic 3 (46편): supervised, label, 모델, semi, semi supervised
   Topic 4 (43편): energy, 에너지, building, heat, power

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

딥러닝 (2021-2022) BERTopi

## 토픽 비교

In [18]:
compare_topics(results)



기간별 토픽 비교

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
전통ML 분야
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2021-2022 주요 토픽:
   Topic 0 (77편): cell, water, concentration, activity, respectively
   Topic 1 (72편): technology, 확인, 제공, 분야, level
   Topic 2 (53편): health, survey, logistic, logistic regression, 회귀분석
   Topic 3 (43편): power, energy, generation, forecasting, distribution
   Topic 4 (33편): detection, 탐지, attack, 공격, maliciou

2023-2025 주요 토픽:
   Topic 0 (96편): health, survey, logistic regression, logistic, 회귀분석
   Topic 1 (50편): detection, security, malware, 탐지, attack
   Topic 2 (46편): robot, unity, reinforcement, virtual, environment
   Topic 3 (46편): supervised, label, 모델, semi, semi supervised
   Topic 4 (43편): energy, 에너지, building, heat, power

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
딥러닝 분야
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

2021-2022 주요 토픽:
   Topic

## 시각화 및 저장

### 전통ML
- 막대 그래프
- 거리 맵
- 계층 구조

In [19]:
from IPython.display import display

# 토픽 키워드 막대 그래프
ai_type = '전통ML'
period = '2021-2022'
model = results[ai_type][period]['model']
print(f"\n📊 {ai_type} ({period}) - Barchart")
display(model.visualize_barchart(top_n_topics=8, n_words=10))

ai_type = '전통ML'
period = '2023-2025'
model = results[ai_type][period]['model']
print(f"\n📊 {ai_type} ({period}) - Barchart")
display(model.visualize_barchart(top_n_topics=8, n_words=10))



📊 전통ML (2021-2022) - Barchart



📊 전통ML (2023-2025) - Barchart


In [20]:
# 토픽 거리 맵
ai_type = '전통ML'
period = '2021-2022'
model = results[ai_type][period]['model']
print(f"\n🗺 {ai_type} ({period}) - Topic Map")
display(model.visualize_topics())

ai_type = '전통ML'
period = '2023-2025'
model = results[ai_type][period]['model']
print(f"\n🗺 {ai_type} ({period}) - Topic Map")
display(model.visualize_topics())


🗺 전통ML (2021-2022) - Topic Map



🗺 전통ML (2023-2025) - Topic Map


In [21]:
# 토픽 계층 구조
ai_type = '전통ML'
period = '2021-2022'
model = results[ai_type][period]['model']
print(f"\n{ai_type} ({period}) - Hierarchy")
display(model.visualize_hierarchy())

ai_type = '전통ML'
period = '2023-2025'
model = results[ai_type][period]['model']
print(f"\n{ai_type} ({period}) - Hierarchy")
display(model.visualize_hierarchy())


전통ML (2021-2022) - Hierarchy



전통ML (2023-2025) - Hierarchy


### 딥러닝
- 막대 그래프
- 거리 맵
- 계층 구조

In [22]:
# 토픽 키워드 막대 그래프
ai_type = '딥러닝'
period = '2021-2022'
model = results[ai_type][period]['model']
print(f"\n{ai_type} ({period}) - Barchart")
display(model.visualize_barchart(top_n_topics=8, n_words=10))

ai_type = '딥러닝'
period = '2023-2025'
model = results[ai_type][period]['model']
print(f"\n{ai_type} ({period}) - Barchart")
display(model.visualize_barchart(top_n_topics=8, n_words=10))


딥러닝 (2021-2022) - Barchart



딥러닝 (2023-2025) - Barchart


In [23]:
# 토픽 거리 맵
ai_type = '딥러닝'
period = '2021-2022'
model = results[ai_type][period]['model']
print(f"\n{ai_type} ({period}) - Topic Map")
display(model.visualize_topics())

ai_type = '딥러닝'
period = '2023-2025'
model = results[ai_type][period]['model']
print(f"\n{ai_type} ({period}) - Topic Map")
display(model.visualize_topics())


딥러닝 (2021-2022) - Topic Map



딥러닝 (2023-2025) - Topic Map


In [24]:
# 토픽 계층 구조
ai_type = '딥러닝'
period = '2021-2022'
model = results[ai_type][period]['model']
print(f"\n🌳 {ai_type} ({period}) - Hierarchy")
display(model.visualize_hierarchy())

ai_type = '딥러닝'
period = '2023-2025'
model = results[ai_type][period]['model']
print(f"\n🌳 {ai_type} ({period}) - Hierarchy")
display(model.visualize_hierarchy())


🌳 딥러닝 (2021-2022) - Hierarchy



🌳 딥러닝 (2023-2025) - Hierarchy


In [25]:
visualize_topics(results, output_dir=OUTPUT_DIR)
print(f"{OUTPUT_DIR} 폴더 안에 시각화 HTML 파일 생성")

save_results(results)


전통ML (2021-2022):
막대 그래프: bertopic_results/전통ML_2021-2022_barchart.html
토픽 맵: bertopic_results/전통ML_2021-2022_topics.html
계층 구조: bertopic_results/전통ML_2021-2022_hierarchy.html

전통ML (2023-2025):
막대 그래프: bertopic_results/전통ML_2023-2025_barchart.html
토픽 맵: bertopic_results/전통ML_2023-2025_topics.html
계층 구조: bertopic_results/전통ML_2023-2025_hierarchy.html

딥러닝 (2021-2022):
막대 그래프: bertopic_results/딥러닝_2021-2022_barchart.html
토픽 맵: bertopic_results/딥러닝_2021-2022_topics.html
계층 구조: bertopic_results/딥러닝_2021-2022_hierarchy.html

딥러닝 (2023-2025):
막대 그래프: bertopic_results/딥러닝_2023-2025_barchart.html
토픽 맵: bertopic_results/딥러닝_2023-2025_topics.html
계층 구조: bertopic_results/딥러닝_2023-2025_hierarchy.html
bertopic_results 폴더 안에 시각화 HTML 파일 생성

결과 저장: bertopic_analysis.xlsx
